# Example Run of LIGNIN-KMC
Written by: Michael Orella <br>
23 January 2019 <br>
Brief examples of how to use the LIGNIN-KMC package within python

In [1]:
%load_ext autoreload
%autoreload 2

Install LIGNIN-KMC

In [2]:
import ligninkmc as kmc
import numpy as np

import scipy.sparse

from ligninkmc.kmc_functions import do_event, update_events
from ligninkmc.kmc_common import Monomer, Event, G, S

## SG Lignin Example

The first step of this example is for us to define the rates of bond formation. This is done by using a dictionary that maps the bond string ('5o4','bo4','b5','55','bb','ao4',etc...) to a dictionary that maps monomer type (0 - coniferyl, 1 - sinapyl, 2 - caffeoyl) to a dictionary that maps fragment sizes ('monomer','dimer') to the transition state energy in kcal/mol. These values can be tuned and updated to reflect new developments in better understanding of lignin chemistry. Once the values have been input, they are converted from units of kcal/mol to units of joule/molecule. From this point, they are used in the Eyring equation to calculate an equivalent dictionary of dictionaries of dictionaries of rates.

$$ r_i = \dfrac{k_BT}{h}\exp\left(-\dfrac{\Delta G_i}{k_BT}\right) $$

In [3]:
kb = 1.38064852e-23 # J / K
h = 6.62607004e-34 # J s
temp = 298.15 #K
kcalToJoule = 4184 / 6.022140857e23

#Energies input in kcal/mol
energies = {'5o4':{(G,G):{('monomer','monomer'):11.2,('monomer','oligomer'):14.6,
                          ('oligomer','monomer'):14.6,('oligomer','oligomer'):4.4},
                   (S,G):{('monomer','monomer'):10.9,('monomer','oligomer'):14.6,
                          ('oligomer','monomer'):14.6,('oligomer','oligomer'):4.4}},
            '55':{(G,G):{('monomer','monomer'):12.5,('monomer','oligomer'):15.6,
                          ('oligomer','monomer'):15.6,('oligomer','oligomer'):3.8}},
            'b5':{(G,G):{('monomer','monomer'):5.5,('monomer','oligomer'):5.8,
                          ('oligomer','monomer'):5.8,('oligomer','oligomer'):5.8},
                  (G,S):{('monomer','monomer'):5.5,('monomer','oligomer'):5.8,
                          ('oligomer','monomer'):5.8,('oligomer','oligomer'):5.8}},
            'bb':{(G,G):{('monomer','monomer'):5.2,('monomer','oligomer'):5.2,('oligomer','monomer'):5.2,('oligomer','oligomer'):5.2},
                  (S,G):{('monomer','monomer'):6.5,('monomer','oligomer'):6.5,('oligomer','monomer'):6.5,('oligomer','oligomer'):6.5},
                  (S,S):{('monomer','monomer'):5.2,('monomer','oligomer'):5.2,('oligomer','monomer'):5.2,('oligomer','oligomer'):5.2}},
            'bo4':{(G,G):{('monomer','monomer'):6.3,('monomer','oligomer'):6.2,
                          ('oligomer','monomer'):6.2,('oligomer','oligomer'):6.2},
                   (S,G):{('monomer','monomer'):9.1,('monomer','oligomer'):6.2,
                          ('oligomer','monomer'):6.2,('oligomer','oligomer'):6.2},
                   (G,S):{('monomer','monomer'):8.9,('monomer','oligomer'):6.2,
                          ('oligomer','monomer'):6.2,('oligomer','oligomer'):6.2},
                   (S,S):{('monomer','monomer'):9.8,('monomer','oligomer'):10.4,
                          ('oligomer','monomer'):10.4,('oligomer','oligomer'):10.4}},
            'ao4':{(G,G):{('monomer','monomer'):20.7,('monomer','oligomer'):20.7,
                          ('oligomer','monomer'):20.7,('oligomer','oligomer'):20.7},
                   (S,G):{('monomer','monomer'):20.7,('monomer','oligomer'):20.7,
                          ('oligomer','monomer'):20.7,('oligomer','oligomer'):20.7},
                   (G,S):{('monomer','monomer'):20.7,('monomer','oligomer'):20.7,
                          ('oligomer','monomer'):20.7,('oligomer','oligomer'):20.7},
                   (S,S):{('monomer','monomer'):20.7,('monomer','oligomer'):20.7,
                          ('oligomer','monomer'):20.7,('oligomer','oligomer'):20.7}},
            'b1':{(G,G):{('monomer','oligomer'):9.6,
                          ('oligomer','monomer'):9.6,('oligomer','oligomer'):9.6},
                  (S,G):{('monomer','oligomer'):11.7,
                          ('oligomer','monomer'):11.7,('oligomer','oligomer'):11.7},
                  (G,S):{('monomer','oligomer'):10.7,
                          ('oligomer','monomer'):10.7,('oligomer','oligomer'):10.7},
                  (S,S):{('monomer','oligomer'):11.9,
                          ('oligomer','monomer'):11.9,('oligomer','oligomer'):11.9}},
            'oxidation':{G:{'monomer':0.9,'oligomer':6.3},S:{'monomer':0.6,'oligomer':2.2}},
            'hydration':{G:{'monomer':11.1,'oligomer':11.1},S:{'monomer':11.7,'oligomer':11.7}}}
energies['bb'][(G,S)] = energies['bb'][(S,G)]

#Correct units of the energies
energiesj = {bond : {monType : {size : energies[bond][monType][size] * kcalToJoule for size in energies[bond][monType]}
                    for monType in energies[bond] } 
            for bond in energies }

#Calculate the rates of reaction
rates = {bond : {monType : { size : kb * temp / h * np.exp ( - energiesj[bond][monType][size] / kb / temp ) 
                            for size in energies[bond][monType]} 
                 for monType in energies[bond] }
         for bond in energies}

With the rates generated, it is now time to initialize the simulation with a set of monomers, and the ratio of S and G units that should be incorporated. 

In [4]:
n = 4

mons = [Monomer(S, i) for i in range(n)]
startEvents = [Event('oxidation', [i], rates['oxidation'][S]['monomer']) for i in range(n)]

When the monomers and starting events have been initialized, they are grouped into the "state" and "events" which are necessary to start the simulation. The final pieces of information needed to run the simulation are the maximum number of monomers that should be studied and the final simulation time. In this case, we choose a simulation time of 1 second, and a maximum number of monomers of 10. Once these parameters are set, we can go about running the simulation. To show the state, we will print the adjacency matrix that has been generated, although this is not the typical output examined.

In [5]:
state = { i : {'monomer' : mons[i] , 'affected' : {startEvents[i]} } for i in range(n) }
events = { startEvents[i] for i in range(n) }

event_dict = {hash(event): event for event in events}
r_vec = {hash(event): event.rate / n for event in events}
adj = scipy.sparse.dok_matrix((n,n))
t = [0,]
np.random.seed(0)

Check running the Gillespie algorithm by hand to make sure that the proper events are being generated

In [46]:
hashes = list(r_vec.keys())
all_rates = list(r_vec.values())
rtot = np.sum(all_rates)

j = np.random.choice(hashes,p = all_rates/rtot)
event = event_dict[j]

print(event)

dt = ( 1 / rtot ) * np.log ( 1 / np.random.rand(1) )
t.extend(t[-1] + dt)

do_event(event,state,adj)

ValueError: a must be non-empty

In [47]:
print([str(state[i]["monomer"]) for i in state])

['0: sinapyl alcohol is connected to {0, 2} and active at position 4', '1: sinapyl alcohol is connected to {1, 3} and active at position 4', '2: sinapyl alcohol is connected to {0, 2} and active at position 4', '3: sinapyl alcohol is connected to {1, 3} and active at position 4']


In [48]:
update_events(state, adj, event, event_dict, r_vec, rates, max_mon=2)

KeyError: 378957238099544

In [45]:
print(list(event_dict.values()))

[]
